# 🔴 HANDWRITTEN DIGIT RECOGNITION
### Deep Learning Powered by CNN

**Project:** Handwritten Digit Recognition System Using Convolutional Neural Networks (CNN)
**Course:** BCA – Artificial Intelligence | **Platform:** Google Colab | **Framework:** TensorFlow / Keras

---

## 1. Project Introduction

This project trains a **Convolutional Neural Network (CNN)** on the **MNIST** dataset (70,000 images of handwritten digits, 28 × 28 pixels, grayscale) and wraps the trained model in a **Gradio** web application.

**What the app can do**
1. 📤 Upload an image of a handwritten digit (PNG / JPG / JPEG)
2. ✏️ Draw a digit on an interactive canvas
3. 🔮 Predict the digit (0–9) with the trained CNN
4. 📊 Show the confidence (%) and the probability of all 10 classes

**Every prediction is made by the real trained model – nothing is simulated.**

**How to run:** `Runtime → Run all` (or run each cell from top to bottom).
Training takes about 2–4 minutes on the free Colab CPU/GPU. A shareable public link is printed in **Section 15**.

| Step | Section |
|---|---|
| Setup | 2 – Import libraries |
| Data | 3 – Load, 4 – Preprocess, 5 – Visualize |
| Model | 6 – Build, 7 – Compile, 8 – Train |
| Evaluation | 9 – Evaluate, 10 – Confusion matrix, 11 – Prediction test |
| Application | 12 – Upload processing, 13 – Drawing canvas, 14 – Gradio app, 15 – Launch |
| Saving | 16 – Save model |

## 2. Import Required Libraries

First we make sure a compatible **Gradio 5.x** is installed (Colab usually has TensorFlow, NumPy, Pillow, Matplotlib and Scikit-learn already). Then we import everything we need.

In [ ]:
# Install a Gradio version this notebook was written for (5.x). Takes ~20 seconds.
!pip install -q "gradio>=5.0,<6.0"

In [ ]:
import os
import random
import warnings

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import confusion_matrix, classification_report

import gradio as gr

warnings.filterwarnings("ignore")

# Make results reproducible
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Black / red / white theme for every Matplotlib chart in this notebook
RED, DARK_RED, BLACK, CHARCOAL, WHITE, GREY = "#E50914", "#B20710", "#0B0B0B", "#171717", "#FFFFFF", "#BDBDBD"
plt.rcParams.update({
    "figure.facecolor": CHARCOAL, "axes.facecolor": BLACK, "savefig.facecolor": CHARCOAL,
    "axes.edgecolor": "#3A3A3A", "axes.labelcolor": WHITE, "text.color": WHITE,
    "xtick.color": GREY, "ytick.color": GREY, "axes.titlecolor": WHITE,
    "grid.color": "#2A2A2A", "axes.prop_cycle": plt.cycler(color=[RED, WHITE, GREY, DARK_RED]),
})

print("TensorFlow :", tf.__version__)
print("Keras      :", keras.__version__)
print("Gradio     :", gr.__version__)
print("GPU found  :", len(tf.config.list_physical_devices("GPU")) > 0)

## 3. Load MNIST Dataset

MNIST is built into Keras, so no manual download is needed. It is already split into **60,000 training** and **10,000 testing** images.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print("Training images shape :", x_train.shape)
print("Training labels shape :", y_train.shape)
print("Testing images shape  :", x_test.shape)
print("Testing labels shape  :", y_test.shape)
print("Pixel value range     :", x_train.min(), "to", x_train.max())

## 4. Data Preprocessing

Neural networks train better with small numbers, so we:
1. **Normalize** pixels from `0–255` to `0–1`.
2. **Reshape** images from `(28, 28)` to `(28, 28, 1)` – the extra `1` is the single grayscale channel the CNN expects.

> ⚠️ The same scaling (white digit on black background, values 0–1) is applied to uploaded and drawn images later, so training and prediction stay consistent.

In [ ]:
# Keep untouched copies for visualization
x_train_raw, x_test_raw = x_train.copy(), x_test.copy()

# 1) Normalize to 0-1
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# 2) Add the channel dimension -> (samples, 28, 28, 1)
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

print("x_train shape :", x_train.shape)
print("x_test shape  :", x_test.shape)
print("Pixel range   :", x_train.min(), "to", x_train.max())

## 5. Data Visualization

Let's look at some sample digits and check how many images exist for each class.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train_raw[i], cmap="gray")
    ax.set_title(f"Label: {y_train[i]}", color=WHITE)
    ax.axis("off")
fig.suptitle("Sample MNIST Training Images", color=RED, fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

counts = np.bincount(y_train)
fig, ax = plt.subplots(figsize=(9, 3.5))
bars = ax.bar(range(10), counts, color=RED, edgecolor=DARK_RED)
ax.set_xticks(range(10))
ax.set_xlabel("Digit class")
ax.set_ylabel("Number of images")
ax.set_title("Training Set Class Distribution", color=WHITE)
ax.bar_label(bars, color=GREY, fontsize=8)
plt.tight_layout()
plt.show()

## 6. Build CNN Model

**Architecture**

| Layer | Purpose |
|---|---|
| `Input (28,28,1)` | One grayscale image |
| Augmentation (small rotate / zoom / shift) | Makes the model more robust to real-world handwriting. **Only active during training.** |
| `Conv2D(32) + ReLU` ×2 | Detect edges and simple strokes |
| `MaxPooling2D` | Shrink the image, keep strong features |
| `Conv2D(64) + ReLU` ×2 | Detect curves and digit parts |
| `MaxPooling2D` | Shrink again |
| `Flatten` | Convert feature maps to a vector |
| `Dense(128) + ReLU` | Combine features |
| `Dropout(0.5)` | Randomly switch off neurons to reduce overfitting |
| `Dense(10) + Softmax` | Probability for each digit 0–9 |

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),

    # Data augmentation (only used while training, ignored during prediction)
    layers.RandomRotation(0.04, fill_mode="constant"),
    layers.RandomZoom(0.1, fill_mode="constant"),
    layers.RandomTranslation(0.08, 0.08, fill_mode="constant"),

    # Convolution block 1
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    # Convolution block 2
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    # Classifier
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(10, activation="softmax"),
], name="digit_cnn")

print("Model built. Input shape:", model.input_shape, "| Output shape:", model.output_shape)

## 7. Compile Model

- **Optimizer – Adam:** adjusts the weights efficiently.
- **Loss – sparse categorical crossentropy:** correct loss for integer labels (0–9) with a Softmax output.
- **Metric – accuracy:** the percentage of correct predictions.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## 8. Train Model

We train for up to **12 epochs** with batch size **128**. 10% of the training data is used as **validation data** to monitor overfitting. `EarlyStopping` stops training automatically when validation loss stops improving and restores the best weights.

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)

history = model.fit(
    x_train, y_train,
    epochs=12,
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=2,
)

# ---- Training vs validation accuracy and loss ----
epochs_range = range(1, len(history.history["accuracy"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(epochs_range, history.history["accuracy"], color=RED, marker="o", label="Training accuracy")
axes[0].plot(epochs_range, history.history["val_accuracy"], color=WHITE, marker="s", label="Validation accuracy")
axes[0].set_title("Model Accuracy"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy")
axes[0].legend(facecolor=CHARCOAL, edgecolor="#3A3A3A"); axes[0].grid(True)

axes[1].plot(epochs_range, history.history["loss"], color=RED, marker="o", label="Training loss")
axes[1].plot(epochs_range, history.history["val_loss"], color=WHITE, marker="s", label="Validation loss")
axes[1].set_title("Model Loss"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
axes[1].legend(facecolor=CHARCOAL, edgecolor="#3A3A3A"); axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"Final training accuracy   : {history.history['accuracy'][-1] * 100:.2f}%")
print(f"Final validation accuracy : {history.history['val_accuracy'][-1] * 100:.2f}%")
print(f"Final training loss       : {history.history['loss'][-1]:.4f}")
print(f"Final validation loss     : {history.history['val_loss'][-1]:.4f}")

## 9. Model Evaluation

Now we test the model on the **10,000 test images it has never seen** during training.

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc * 100:.2f}%")

## 10. Confusion Matrix and Classification Report

- The **confusion matrix** shows which digits the model confuses (rows = true digit, columns = predicted digit). A strong diagonal means good performance.
- The **classification report** gives precision, recall and F1-score for every digit.

In [ ]:
y_prob = model.predict(x_test, batch_size=256, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

cm = confusion_matrix(y_test, y_pred)

from matplotlib.colors import LinearSegmentedColormap
black_red = LinearSegmentedColormap.from_list("black_red", [BLACK, DARK_RED, RED])

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap=black_red)
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel("Predicted digit"); ax.set_ylabel("True digit")
ax.set_title("Confusion Matrix – MNIST Test Set", color=WHITE)
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha="center", va="center", color=WHITE, fontsize=9)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

print("Classification Report\n" + "=" * 60)
print(classification_report(y_test, y_pred, digits=4))
print(f"Misclassified images: {int((y_pred != y_test).sum())} out of {len(y_test)}")

## 11. Prediction Testing

We pick 10 random test images and compare the model's prediction with the real label.
White title = correct, **red title = wrong**.

In [ ]:
idx = np.random.choice(len(x_test), 10, replace=False)

fig, axes = plt.subplots(2, 5, figsize=(12, 5.5))
for ax, i in zip(axes.flat, idx):
    probs = model.predict(x_test[i:i + 1], verbose=0)[0]
    pred, conf = int(np.argmax(probs)), float(np.max(probs)) * 100
    ax.imshow(x_test_raw[i], cmap="gray")
    ax.set_title(f"True: {y_test[i]} | Pred: {pred}\n{conf:.1f}%",
                 color=WHITE if pred == y_test[i] else RED, fontsize=10)
    ax.axis("off")
fig.suptitle("Random Test Predictions", color=RED, fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

## 12. Image Upload Processing

Real photos and drawings do not look like MNIST, so every uploaded image goes through **the same pipeline**:

1. Load the image and convert it to **grayscale** (transparent PNGs are placed on a white background).
2. **Detect the background:** if the image is mostly bright (white paper, black ink) it is **inverted**, because MNIST digits are *white on black*.
3. Remove the background level and noise, then scale to the range 0–1.
4. **Crop** tightly around the digit.
5. **Resize** the digit so its longest side is 20 pixels (keeping the aspect ratio) – exactly how MNIST was made.
6. Paste it into a **28 × 28** black canvas and **center it by its center of mass**.
7. The result has shape `(28, 28)`, values 0–1 – ready for the CNN.

In [ ]:
MNIST_SIZE = 28   # final image size expected by the CNN
DIGIT_BOX = 20    # the digit itself fits inside a 20x20 box (MNIST convention)
ALLOWED_EXTENSIONS = (".png", ".jpg", ".jpeg")


def pil_to_gray(img):
    """Convert any PIL image to a float32 grayscale array (0-255). Transparent areas become white."""
    if img.mode in ("RGBA", "LA") or (img.mode == "P" and "transparency" in img.info):
        rgba = img.convert("RGBA")
        white = Image.new("RGBA", rgba.size, (255, 255, 255, 255))
        img = Image.alpha_composite(white, rgba)
    return np.array(img.convert("L"), dtype=np.float32)


def shift_image(arr, dy, dx):
    """Move an image dy rows down and dx columns right, filling the gap with zeros."""
    h, w = arr.shape
    out = np.zeros_like(arr)
    dst_y = slice(max(dy, 0), min(h, h + dy))
    src_y = slice(max(-dy, 0), min(h, h - dy))
    dst_x = slice(max(dx, 0), min(w, w + dx))
    src_x = slice(max(-dx, 0), min(w, w - dx))
    out[dst_y, dst_x] = arr[src_y, src_x]
    return out


def preprocess_digit(img):
    """Convert any PIL image of a handwritten digit into a MNIST-style (28, 28) float32 array in 0-1."""
    gray = pil_to_gray(img)

    # Handle white-background AND black-background images
    if np.median(gray) > 127:
        gray = 255.0 - gray                       # dark ink on light paper -> make the digit bright

    # Remove background level / noise
    gray = np.clip(gray - np.median(gray), 0, None)
    if gray.max() < 25:
        raise ValueError("No handwritten digit was found in the image. Please use a clearer image.")
    gray = gray / gray.max()                      # scale to 0-1
    gray[gray < 0.15] = 0.0                       # drop faint noise

    # Crop tightly around the digit
    rows = np.where(gray.max(axis=1) > 0)[0]
    cols = np.where(gray.max(axis=0) > 0)[0]
    crop = gray[rows[0]:rows[-1] + 1, cols[0]:cols[-1] + 1]

    # Resize so the longest side is 20 px (keep aspect ratio)
    h, w = crop.shape
    scale = DIGIT_BOX / max(h, w)
    new_w, new_h = max(1, int(round(w * scale))), max(1, int(round(h * scale)))
    digit = Image.fromarray((crop * 255).astype(np.uint8)).resize(
        (new_w, new_h), Image.Resampling.LANCZOS)

    # Paste into the centre of a 28x28 black canvas
    canvas = np.zeros((MNIST_SIZE, MNIST_SIZE), dtype=np.float32)
    top, left = (MNIST_SIZE - new_h) // 2, (MNIST_SIZE - new_w) // 2
    canvas[top:top + new_h, left:left + new_w] = np.array(digit, dtype=np.float32) / 255.0

    # Centre by centre of mass (like the original MNIST dataset)
    total = canvas.sum()
    if total > 0:
        ys, xs = np.indices(canvas.shape)
        cy, cx = (ys * canvas).sum() / total, (xs * canvas).sum() / total
        canvas = shift_image(canvas, int(round(13.5 - cy)), int(round(13.5 - cx)))

    return np.clip(canvas, 0, 1).astype(np.float32)


def load_uploaded_image(file_path):
    """Validate and open an uploaded file. Raises ValueError with a friendly message if invalid."""
    if file_path is None:
        raise ValueError("No image uploaded. Please upload a PNG, JPG or JPEG image of a digit.")
    if not str(file_path).lower().endswith(ALLOWED_EXTENSIONS):
        raise ValueError("Unsupported file type. Please upload a PNG, JPG or JPEG image.")
    try:
        img = Image.open(file_path)
        img.load()
        return img
    except Exception:
        raise ValueError("The file could not be read as an image. Please upload a valid image.")


def predict_digit_array(processed):
    """Run the trained CNN on a (28, 28) array. Returns the 10 Softmax probabilities."""
    x = processed.reshape(1, MNIST_SIZE, MNIST_SIZE, 1).astype("float32")   # same shape as training data
    return model(x, training=False).numpy()[0]


print("Image preprocessing functions are ready.")

In [ ]:
# ---- Self-test: the SAME preprocessing must work for MNIST-like images in any style ----
# Take real test digits, turn them into big "photo-like" images (white paper, black ink)
# and check that the pipeline + CNN still recognise them.
correct = 0
for i in range(20):
    big = Image.fromarray(x_test_raw[i]).resize((220, 220), Image.Resampling.BICUBIC)
    white_bg = Image.fromarray(255 - np.array(big))             # black ink on white paper
    black_bg = big                                              # white ink on black
    for variant in (white_bg, black_bg):
        probs = predict_digit_array(preprocess_digit(variant))
        correct += int(np.argmax(probs) == y_test[i])
print(f"Pipeline self-test: {correct}/40 correct (white-background and black-background variants)")

# Show what the pipeline produces
fig, axes = plt.subplots(1, 3, figsize=(9, 3.4))
sample = Image.fromarray(255 - np.array(Image.fromarray(x_test_raw[0]).resize((220, 220))))
axes[0].imshow(sample, cmap="gray"); axes[0].set_title("Uploaded (white background)")
axes[1].imshow(preprocess_digit(sample), cmap="gray"); axes[1].set_title("Processed 28x28")
axes[2].imshow(x_test_raw[0], cmap="gray"); axes[2].set_title("Original MNIST")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

## 13. Interactive Drawing Canvas

The web app uses Gradio's **ImageEditor** as a drawing pad: a white canvas with a fixed **black brush**. The editor returns the drawing as a dictionary; we flatten it onto white, check it is not empty, and then send it through the **same `preprocess_digit()` pipeline** used for uploads.

In [ ]:
CANVAS_SIZE = 300


def blank_canvas():
    """A fresh white canvas value for the Gradio ImageEditor (used at start, on Clear and on Reset)."""
    white = np.full((CANVAS_SIZE, CANVAS_SIZE, 3), 255, dtype=np.uint8)
    return {"background": white, "layers": [], "composite": None}


def canvas_to_pil(editor_value):
    """Convert the ImageEditor output into a grayscale PIL image. Raises ValueError if nothing is drawn."""
    if editor_value is None:
        raise ValueError("The canvas is empty. Please draw a digit first.")

    # ImageEditor returns {"background":..., "layers":[...], "composite":...}
    comp = editor_value.get("composite") if isinstance(editor_value, dict) else editor_value
    if comp is None:
        raise ValueError("The canvas is empty. Please draw a digit first.")

    arr = np.asarray(comp)
    if arr.ndim == 3 and arr.shape[2] == 4:            # RGBA -> flatten onto white
        alpha = arr[:, :, 3:4].astype(np.float32) / 255.0
        rgb = arr[:, :, :3].astype(np.float32)
        arr = rgb * alpha + 255.0 * (1.0 - alpha)
    if arr.ndim == 3:
        arr = arr[:, :, :3].mean(axis=2)
    arr = np.clip(arr, 0, 255).astype(np.uint8)

    if (arr < 150).sum() < 30:                         # almost no dark pixels -> nothing drawn
        raise ValueError("The canvas is empty. Please draw a digit first.")
    return Image.fromarray(arr, mode="L")


print("Canvas helper functions are ready.")

In [ ]:
# ---- Self-test: simulate a digit drawn on the canvas (black strokes on a transparent RGBA layer) ----
demo_layer = Image.new("RGBA", (CANVAS_SIZE, CANVAS_SIZE), (0, 0, 0, 0))
d = ImageDraw.Draw(demo_layer)
d.line([(150, 50), (80, 190), (230, 190)], fill=(0, 0, 0, 255), width=14)   # roughly the top of a "4"
d.line([(190, 60), (190, 260)], fill=(0, 0, 0, 255), width=14)             # a "4"
fake_editor_value = {"background": blank_canvas()["background"], "layers": [], "composite": np.array(demo_layer)}

probs = predict_digit_array(preprocess_digit(canvas_to_pil(fake_editor_value)))
print("Simulated canvas drawing -> predicted:", int(np.argmax(probs)), f"({probs.max() * 100:.1f}%)")

try:
    canvas_to_pil(blank_canvas())
except ValueError as e:
    print("Empty canvas correctly rejected:", e)

## 14. Gradio Web Application

Here we build the dashboard. It has:
- a **header** with a neural-network graphic,
- two cards: **Upload Handwritten Digit** and **Draw Your Digit**,
- a **Prediction Result** section (big digit, confidence %, all 10 probabilities, bar chart, processed image),
- a **Reset** button and a **Model Information** section.

The theme uses only black `#0B0B0B`, charcoal `#171717`, red `#E50914` / `#B20710` and white. Errors are shown as friendly red messages instead of crashing the app.

**Part A** defines the result-display helpers, **Part B** builds the interface.

In [ ]:
# ---------------- Part A: result display helpers ----------------

def build_network_svg():
    """A small decorative neural-network graphic for the header."""
    xs = [20, 75, 130, 185]
    counts = [4, 5, 5, 3]
    pts = [[(x, 50 + (i - (c - 1) / 2) * 20 + 0) for i in range(c)] for x, c in zip(xs, counts)]
    lines, circles = "", ""
    for a, b in zip(pts[:-1], pts[1:]):
        for (x1, y1) in a:
            for (x2, y2) in b:
                lines += f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" stroke="#E50914" stroke-opacity="0.35" stroke-width="1"/>'
    for k, layer in enumerate(pts):
        for (x, y) in layer:
            fill = "#E50914" if k in (0, 3) else "#FFFFFF"
            circles += f'<circle cx="{x}" cy="{y}" r="5" fill="{fill}"/>'
    return f'<svg width="210" height="100" viewBox="0 0 210 100" xmlns="http://www.w3.org/2000/svg">{lines}{circles}</svg>'


def error_html(message):
    return (
        '<div style="background:#2a0a0c;border:1px solid #E50914;border-radius:12px;padding:18px;'
        'text-align:center;">'
        '<div style="font-size:30px;">⚠️</div>'
        f'<div style="color:#FFFFFF;font-weight:700;margin-top:6px;">{message}</div>'
        '<div style="color:#BDBDBD;font-size:13px;margin-top:6px;">Fix the input and press the Predict button again.</div>'
        '</div>'
    )


PLACEHOLDER_HTML = (
    '<div style="background:#0B0B0B;border:1px dashed #3A3A3A;border-radius:12px;padding:30px;text-align:center;">'
    '<div style="color:#BDBDBD;font-size:13px;letter-spacing:2px;">PREDICTED DIGIT</div>'
    '<div style="color:#3A3A3A;font-size:96px;font-weight:800;line-height:1.1;">?</div>'
    '<div style="color:#BDBDBD;">Upload or draw a digit, then press <b style="color:#E50914;">Predict</b>.</div>'
    '</div>'
)


def result_html(probs):
    """Big predicted digit + confidence + the probability of all 10 digits."""
    digit = int(np.argmax(probs))
    conf = float(probs[digit]) * 100

    cells = ""
    for i, p in enumerate(probs):
        top = i == digit
        cells += (
            f'<div style="background:{"#E50914" if top else "#0B0B0B"};border:1px solid '
            f'{"#E50914" if top else "#2A2A2A"};border-radius:10px;padding:8px 4px;text-align:center;">'
            f'<div style="color:#FFFFFF;font-weight:800;font-size:18px;">{i}</div>'
            f'<div style="color:{"#FFFFFF" if top else "#BDBDBD"};font-size:12px;">{p * 100:.2f}%</div></div>'
        )

    return (
        '<div style="background:#0B0B0B;border:1px solid #2A2A2A;border-radius:14px;padding:22px;text-align:center;">'
        '<div style="color:#BDBDBD;font-size:13px;letter-spacing:3px;">PREDICTED DIGIT</div>'
        f'<div style="color:#E50914;font-size:120px;font-weight:900;line-height:1.05;'
        f'text-shadow:0 0 28px rgba(229,9,20,0.55);">{digit}</div>'
        f'<div style="color:#FFFFFF;font-size:22px;font-weight:700;">Confidence: '
        f'<span style="color:#E50914;">{conf:.2f}%</span></div>'
        '<div style="background:#2A2A2A;border-radius:99px;height:10px;margin:14px auto 18px;max-width:320px;">'
        f'<div style="background:linear-gradient(90deg,#B20710,#E50914);width:{conf:.1f}%;height:10px;border-radius:99px;"></div></div>'
        '<div style="color:#BDBDBD;font-size:12px;letter-spacing:2px;margin-bottom:8px;">SCORES FOR ALL DIGITS</div>'
        f'<div style="display:grid;grid-template-columns:repeat(5,1fr);gap:8px;">{cells}</div>'
        '</div>'
    )


def make_confidence_chart(probs):
    """Bar chart of the 10 Softmax probabilities (predicted digit in red)."""
    digit = int(np.argmax(probs))
    fig, ax = plt.subplots(figsize=(6.4, 3.8))
    colors = [RED if i == digit else "#4A4A4A" for i in range(10)]
    bars = ax.bar(range(10), probs * 100, color=colors, edgecolor=BLACK)
    ax.set_xticks(range(10))
    ax.set_ylim(0, 112)
    ax.set_xlabel("Digit"); ax.set_ylabel("Confidence (%)")
    ax.set_title("Confidence Distribution (0-9)", color=WHITE)
    ax.bar_label(bars, labels=[f"{p * 100:.1f}" for p in probs], color=GREY, fontsize=8, padding=2)
    ax.grid(axis="y", alpha=0.4)
    plt.tight_layout()
    plt.close(fig)          # stop Colab from also showing it under the cell
    return fig


def processed_preview(processed):
    """Enlarge the 28x28 processed image so it is easy to see in the app."""
    return Image.fromarray((processed * 255).astype(np.uint8)).resize((224, 224), Image.Resampling.NEAREST)


def run_prediction(pil_image):
    """Shared by upload and canvas: preprocess -> CNN -> (result card, chart, processed image)."""
    processed = preprocess_digit(pil_image)
    probs = predict_digit_array(processed)
    return result_html(probs), make_confidence_chart(probs), processed_preview(processed)


def predict_uploaded(file_path):
    try:
        return run_prediction(load_uploaded_image(file_path))
    except ValueError as e:
        return error_html(str(e)), None, None
    except Exception:
        return error_html("Something went wrong while processing the image. Please try another one."), None, None


def predict_drawn(editor_value):
    try:
        return run_prediction(canvas_to_pil(editor_value))
    except ValueError as e:
        return error_html(str(e)), None, None
    except Exception:
        return error_html("Something went wrong while processing the drawing. Please clear the canvas and try again."), None, None


def reset_all():
    """Reset everything for a new prediction."""
    return None, blank_canvas(), PLACEHOLDER_HTML, None, None


def clear_canvas():
    return blank_canvas()


print("Helper functions are ready.")

In [ ]:
# ---------------- Part B: build the Gradio interface ----------------

CSS = """
:root, .dark, body, .gradio-container {
  --body-background-fill:#0B0B0B; --background-fill-primary:#171717; --background-fill-secondary:#0B0B0B;
  --block-background-fill:#171717; --block-border-color:#2A2A2A; --border-color-primary:#2A2A2A;
  --body-text-color:#FFFFFF; --body-text-color-subdued:#BDBDBD;
  --block-label-text-color:#BDBDBD; --block-title-text-color:#FFFFFF;
  --button-primary-background-fill:#E50914; --button-primary-background-fill-hover:#B20710;
  --button-primary-text-color:#FFFFFF; --button-primary-border-color:#E50914;
  --button-secondary-background-fill:#171717; --button-secondary-background-fill-hover:#222222;
  --button-secondary-text-color:#FFFFFF; --button-secondary-border-color:#E50914;
  --input-background-fill:#0B0B0B; --color-accent:#E50914; --color-accent-soft:#2a0a0c;
}
.gradio-container { background:#0B0B0B !important; max-width:1180px !important; margin:0 auto !important; }
footer { display:none !important; }

.hero { display:flex; align-items:center; justify-content:space-between; gap:20px; flex-wrap:wrap;
  background:linear-gradient(135deg,#171717 0%,#0B0B0B 60%,#2a0a0c 100%);
  border:1px solid #2A2A2A; border-bottom:3px solid #E50914; border-radius:18px; padding:26px 32px; margin-bottom:14px; }
.hero h1 { margin:0; color:#FFFFFF; font-size:34px; font-weight:900; letter-spacing:2px; }
.hero h1 span { color:#E50914; }
.hero p { margin:6px 0 0 0; color:#BDBDBD; font-size:16px; letter-spacing:1px; }
.badge { display:inline-block; background:#2a0a0c; color:#E50914; border:1px solid #E50914; border-radius:99px;
  padding:3px 12px; font-size:12px; letter-spacing:2px; margin-bottom:10px; }

.steps { display:grid; grid-template-columns:repeat(3,1fr); gap:12px; margin-bottom:14px; }
.step { background:#171717; border:1px solid #2A2A2A; border-radius:14px; padding:14px 16px; color:#BDBDBD; font-size:14px; }
.step b { color:#FFFFFF; }
.step .n { color:#E50914; font-weight:900; font-size:20px; margin-right:6px; }

.card { background:#171717 !important; border:1px solid #2A2A2A !important; border-radius:16px !important; padding:18px !important; }
.sec-title { color:#FFFFFF; font-size:17px; font-weight:800; letter-spacing:1px; border-left:4px solid #E50914;
  padding-left:10px; margin-bottom:8px; }
.sec-note { color:#BDBDBD; font-size:13px; margin:0 0 8px 14px; }

button.primary, button.secondary { border-radius:12px !important; font-weight:700 !important; transition:all .2s ease; }
button.primary { background:#E50914 !important; color:#FFFFFF !important; border:1px solid #E50914 !important; }
button.secondary { background:#171717 !important; color:#FFFFFF !important; border:1px solid #E50914 !important; }
button.primary:hover, button.secondary:hover { box-shadow:0 0 18px rgba(229,9,20,0.6) !important; transform:translateY(-1px); }
button.primary:hover { background:#B20710 !important; }

.info-grid { display:grid; grid-template-columns:repeat(auto-fit,minmax(170px,1fr)); gap:12px; }
.info-item { background:#0B0B0B; border:1px solid #2A2A2A; border-radius:12px; padding:14px; }
.info-item .k { color:#BDBDBD; font-size:12px; letter-spacing:2px; }
.info-item .v { color:#FFFFFF; font-size:16px; font-weight:700; margin-top:4px; }
.info-item .v em { color:#E50914; font-style:normal; }
@media (max-width:760px) { .steps { grid-template-columns:1fr; } .hero h1 { font-size:24px; } }
"""

theme = gr.themes.Base(
    primary_hue=gr.themes.colors.red,
    neutral_hue=gr.themes.colors.neutral,
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"],
)

HEADER_HTML = f"""
<div class="hero">
  <div>
    <div class="badge">AI · NEURAL NETWORK · MNIST</div>
    <h1>HANDWRITTEN DIGIT <span>RECOGNITION</span></h1>
    <p>Deep Learning Powered by CNN</p>
  </div>
  <div>{build_network_svg()}</div>
</div>
<div class="steps">
  <div class="step"><span class="n">1</span><b>Choose a method.</b> Upload a photo of a digit, or draw one with your mouse or finger.</div>
  <div class="step"><span class="n">2</span><b>Press Predict.</b> Use one clear digit; dark ink on light paper or light ink on dark both work.</div>
  <div class="step"><span class="n">3</span><b>Read the result.</b> See the digit, the confidence and the scores for 0–9. Use Reset to start again.</div>
</div>
"""

MODEL_INFO_HTML = f"""
<div class="sec-title">Model Information</div>
<div class="info-grid">
  <div class="info-item"><div class="k">DATASET</div><div class="v">MNIST</div></div>
  <div class="info-item"><div class="k">ALGORITHM</div><div class="v">Convolutional Neural Network</div></div>
  <div class="info-item"><div class="k">FRAMEWORK</div><div class="v">TensorFlow / Keras</div></div>
  <div class="info-item"><div class="k">INPUT DIMENSIONS</div><div class="v">28 × 28 <em>(grayscale)</em></div></div>
  <div class="info-item"><div class="k">OUTPUT CLASSES</div><div class="v">10 <em>(digits 0–9)</em></div></div>
  <div class="info-item"><div class="k">PARAMETERS</div><div class="v">{model.count_params():,}</div></div>
  <div class="info-item"><div class="k">TEST ACCURACY</div><div class="v"><em>{test_acc * 100:.2f}%</em></div></div>
</div>
"""

with gr.Blocks(theme=theme, css=CSS, title="Handwritten Digit Recognition") as demo:

    gr.HTML(HEADER_HTML)

    # ---------- Main dashboard: two input sections ----------
    with gr.Row(equal_height=False):
        with gr.Column(elem_classes="card"):
            gr.HTML('<div class="sec-title">01 · Upload Handwritten Digit</div>'
                    '<p class="sec-note">Accepted formats: PNG, JPG, JPEG</p>')
            upload_input = gr.Image(type="filepath", sources=["upload"], height=320,
                                    label="Uploaded image")
            predict_upload_btn = gr.Button("🔮 Predict Uploaded Digit", variant="primary")

        with gr.Column(elem_classes="card"):
            gr.HTML('<div class="sec-title">02 · Draw Your Digit</div>'
                    '<p class="sec-note">Draw one large digit in the centre of the white canvas.</p>')
            canvas_input = gr.ImageEditor(
                value=blank_canvas(),
                type="numpy",
                label="Drawing canvas",
                canvas_size=(CANVAS_SIZE, CANVAS_SIZE),
                sources=[],
                layers=False,
                transforms=[],
                brush=gr.Brush(default_size=14, colors=["#000000"], default_color="#000000", color_mode="fixed"),
                eraser=gr.Eraser(default_size=20),
                height=360,
            )
            with gr.Row():
                clear_btn = gr.Button("🧹 Clear Canvas", variant="secondary")
                predict_draw_btn = gr.Button("🔮 Predict Digit", variant="primary")

    # ---------- Prediction result section ----------
    gr.HTML('<div class="sec-title" style="margin-top:18px;">Prediction Result</div>')
    with gr.Row(equal_height=False):
        with gr.Column(elem_classes="card", scale=1):
            result_box = gr.HTML(PLACEHOLDER_HTML)
            processed_out = gr.Image(label="Processed input (28 × 28, what the CNN sees)",
                                     interactive=False, height=240, type="pil")
        with gr.Column(elem_classes="card", scale=1):
            chart_out = gr.Plot(label="Confidence chart")
    reset_btn = gr.Button("↺ Reset / New Prediction", variant="secondary")

    # ---------- Model information ----------
    with gr.Column(elem_classes="card"):
        gr.HTML(MODEL_INFO_HTML)

    # ---------- Button actions ----------
    outputs = [result_box, chart_out, processed_out]
    predict_upload_btn.click(predict_uploaded, inputs=upload_input, outputs=outputs)
    predict_draw_btn.click(predict_drawn, inputs=canvas_input, outputs=outputs)
    clear_btn.click(clear_canvas, inputs=None, outputs=canvas_input)
    reset_btn.click(reset_all, inputs=None,
                    outputs=[upload_input, canvas_input, result_box, chart_out, processed_out])

print("Gradio interface built successfully.")

## 15. Launch Application

`share=True` creates a **public Gradio link** (`https://xxxx.gradio.live`) that you can open on any device – ideal for your college demonstration.

- The app keeps running as long as this Colab session is connected.
- The link is temporary; run this cell again to get a new one.
- To stop the app, run `demo.close()`.

In [ ]:
demo.queue()
demo.launch(share=True, debug=False)

### How to demonstrate
1. Open the **public link** printed above (or use the app displayed below the cell).
2. **Upload:** choose a PNG/JPG image of a handwritten digit → click **Predict Uploaded Digit**.
3. **Draw:** draw a big digit on the white canvas → click **Predict Digit**. Use **Clear Canvas** to redraw.
4. Show the **predicted digit, confidence %, the 10 class scores, the bar chart and the processed 28 × 28 input**.
5. Press **Reset / New Prediction** between examples. Try an empty canvas to show the error message.

## 16. Save Trained Model

The trained model is saved in the modern **Keras format (`.keras`)**. We reload it and confirm it gives identical predictions, so you know the saved file is valid.

In [ ]:
MODEL_PATH = "handwritten_digit_cnn.keras"
model.save(MODEL_PATH)
print(f"Model saved as '{MODEL_PATH}'  ({os.path.getsize(MODEL_PATH) / 1024:.0f} KB)")

# Verify the saved model
reloaded = keras.models.load_model(MODEL_PATH)
same = np.allclose(model.predict(x_test[:100], verbose=0),
                   reloaded.predict(x_test[:100], verbose=0), atol=1e-5)
print("Reloaded model gives identical predictions:", same)

# Optional: download the model to your computer (works in Google Colab)
try:
    from google.colab import files
    files.download(MODEL_PATH)
except ImportError:
    print("Not running in Colab – the model file is in the current folder.")

---
## ✅ Project Summary

| Item | Details |
|---|---|
| Dataset | MNIST – 60,000 training / 10,000 testing images |
| Model | CNN: 4 × Conv2D + 2 × MaxPooling + Dense + Dropout + Softmax(10) |
| Input | 28 × 28 grayscale (normalized 0–1) |
| Result | ~99% test accuracy (see Section 9 for your exact value) |
| Application | Gradio dashboard with image upload + drawing canvas |
| Output | Predicted digit, confidence %, scores for all 10 classes, confidence chart, processed image |
| Saved model | `handwritten_digit_cnn.keras` |

### Troubleshooting
- **`ModuleNotFoundError: gradio`** → run the `pip install` cell in Section 2 again.
- **No public link / `share` error** → check your internet connection and re-run Section 15.
- **Wrong prediction on a photo** → use a single digit, thick dark ink, even lighting and little empty space.
- **Variables missing (`model`, `test_acc`)** → run the notebook from the top with `Runtime → Run all`.

### Future improvements
Recognizing multi-digit numbers, adding letters (EMNIST), and deploying on Hugging Face Spaces.